In [7]:
import sqlite3


db_file_path = 'generated_data.db'

conn = sqlite3.connect(db_file_path)
print(f"Successfully connected to {db_file_path}")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
for table in tables:
    print(type(table[0]))
    if(table[0] not in ["event", "payment", "session", "ticket", "station"]):
        drop_table_sql = f"DROP TABLE IF EXISTS {table[0]}"
        print(table[0])
        cursor.execute(drop_table_sql)


conn.close()

Successfully connected to generated_data.db
<class 'str'>
<class 'str'>
<class 'str'>
tariff
<class 'str'>
discount_rule
<class 'str'>
voucher
<class 'str'>
payment_discount
<class 'str'>
payment_voucher
<class 'str'>
<class 'str'>
<class 'str'>


In [23]:
from faker import Faker
import random

fake = Faker()
payments = []
payment_methods = ['cash', 'card', 'online']
num_rows = 1000
for _ in range(num_rows):
    method = random.choice(payment_methods)
        
    processor_ref = None
    if method != 'cash':
        processor_ref = f"TXN-{fake.unique.bothify(text='????####').upper()}"
        
        
    created_at_dt = fake.date_time_between(start_date='-1y', end_date='now')

    payment = (
            random.randint(1000, 9999),  # session_id
            random.randint(1, 15),       # station_id
            method,
            random.randint(100, 15000),  # amount_cents (e.g., 1.00 to 150.00)
            random.choice([0, 1]),       # approved (0=False, 1=True)
            processor_ref,
            created_at_dt.isoformat()    # created_at
    )
    payments.append(payment)

sql = ''' INSERT INTO payment(session_id, station_id, method, amount_cents, approved, processor_ref, created_at)
              VALUES(?,?,?,?,?,?,?) '''

conn = sqlite3.connect(db_file_path)

cursor = conn.cursor()
cursor.executemany(sql, payments)
conn.commit()



conn.close()

In [28]:
fake = Faker()
stations = []
station_kinds = ['entry_terminal', 'exit_terminal', 'pof']
pof_labels = ["POF-01", "POF-02"]
# "Entry Lane A", "Exit Lane A"

num_rows = 900
for _ in range(num_rows):
    kind = random.choice(station_kinds)
    if(kind == 'pof'):
        label = random.choice(pof_labels)
    elif kind == "entry_terminal":
        label = "Entry Lane A"
    else:
        label = "Exit Lane A"

    station = (
            random.randint(1, 4),       
            kind,
            label
    )
    stations.append(station)

sql = ''' INSERT INTO station(zone_id, kind, label)
              VALUES(?,?,?)'''

conn = sqlite3.connect(db_file_path)

cursor = conn.cursor()
cursor.executemany(sql, stations)
conn.commit()



conn.close()